# CanSat 2026 — post-flight analysis

**Team CAN-Team-25.** The rulebook allows four hours after the launch. This notebook is the
analysis, written and tested before the flight so that on the day it only has to be pointed
at the real logs.

**On launch day, in order:**

1. Copy the SD card's `FLIGHT.CSV` off the card, then extract it:
   `python tools/read_flight_log.py FLIGHT.CSV --out-dir analysis-input`
2. Copy the ground station's `logs/telemetry.csv`.
3. Edit the **Configuration** cell below — the two paths, and the mass if the vehicle was weighed.
4. *Run All.* Every figure and `summary.md` are written to `OUTPUT_DIR`.

**No Jupyter?** The same analysis runs in one command, and produces the same figures and summary:

```
python analysis/flight_analysis.py analysis-input/flight-1.csv --compare logs/telemetry.csv --mass 0.50 --out analysis-output
```

> **Until the launch, this notebook reads a SYNTHETIC flight** from `test-data/synthetic-flight/`,
> generated by `analysis/synthetic_flight.py`. Every number it shows before the paths are changed
> is from that simulation, not from the vehicle.

## Configuration

In [ ]:
from pathlib import Path

# ---- change these on launch day ---------------------------------------------------------
# The onboard SD log (tools/read_flight_log.py output), the ground station's CSV, or a file
# of raw packets. The SD log is the complete record; use it as PRIMARY when you have it.
PRIMARY_LOG = "test-data/synthetic-flight/sd-flight.csv"        # SYNTHETIC until changed
COMPARE_LOG = "test-data/synthetic-flight/ground-telemetry.csv"  # or None

# Flight mass in kg. Needed only for the drag coefficient. The submitted mass was not
# recorded -- weigh the vehicle and put the number here, or leave None.
MASS_KG = 0.500
CANOPY_DIAMETER_M = 0.80

OUTPUT_DIR = "analysis-output"

In [ ]:
import sys

# Find the repository root whether the notebook was opened from the repo root or from analysis/.
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "analysis" / "flight_analysis.py").exists())
sys.path.insert(0, str(ROOT / "analysis"))

import matplotlib.pyplot as plt
import flight_analysis as fa

def repo_path(p):
    p = Path(p)
    return p if p.is_absolute() else ROOT / p

flight = fa.load(repo_path(PRIMARY_LOG))
other = fa.load(repo_path(COMPARE_LOG)) if COMPARE_LOG else None
an = fa.analyse(flight, mass_kg=MASS_KG, canopy_diameter_m=CANOPY_DIAMETER_M, compare=other)
print(f"{len(flight)} samples, format '{flight.kind}', team {flight.team}")

## 1 · Is the data what we think it is?

Read this before any graph. A gap in packet numbers is radio loss or a reset; a note about
several power cycles means the log was not emptied before the flight, and the session that
climbed highest was chosen.

In [ ]:
for key, value in an.quality.items():
    if key != "notes":
        print(f"{key:28s} {value}")
for note in an.quality["notes"]:
    print("NOTE:", note)

## 2 · Flight phases

Found from the barometric altitude with the firmware's own thresholds — a 15 m climb for launch,
a descent faster than 2 m/s held for a second for release. The vehicle's own `FLIGHT` is
declared on the climb and `LANDED` three seconds after it comes to rest, so they are printed
beside the physical events rather than used as them.

In [ ]:
ph = an.phases
for name in ("launch_t", "flight_declared_t", "apex_t", "release_t", "landing_t", "landed_declared_t"):
    value = getattr(ph, name)
    print(f"{name:20s} {'-' if value is None else f'{value:8.2f} s'}")
print(f"apex {ph.apex_altitude} m above the pad, as the vehicle measured it")
fa.plot_altitude(an, full=True)
plt.show()

## 3 · The three mandatory graphs

Altitude, temperature and pressure against time (packet number on the top axis). These are
the vehicle's own transmitted values, cropped to the flight; the phase shading is lift and
hover, then descent.

In [ ]:
fa.plot_altitude(an); plt.show()
fa.plot_temperature(an); plt.show()
fa.plot_pressure(an); plt.show()

## 4 · Descent rate, and the drag coefficient it implies

**The rate is taken from temperature-corrected height, not from the vehicle's altitude.** The
firmware converts pressure with the ISA formula, which assumes 15 °C; on a hot launch day real
height per pascal is several percent larger, so the vehicle's altitude — and any rate taken from
it — reads low. Both are shown. The rulebook cap is 5 m/s.

With the mass and the canopy diameter, the steady rate gives the **drag coefficient** — the one
number the parachute sizing rested on and nobody had measured.

In [ ]:
for key, value in an.descent.items():
    print(f"{key:40s} {value}")
fa.plot_descent(an)
plt.show()

## 5 · Acceleration, orientation and stability

**Every peak here is a lower bound.** The log has one row per packet — about three a second in
flight — and the canopy snatch and the landing impact last tens of milliseconds. The minimum
acceleration just after release is the more trustworthy number: free fall reads near zero.

Yaw is relative (the IMU has no magnetometer), but its *rate* — the spin under the canopy — is
still a real measurement. The pendulum frequency is only trustworthy below the Nyquist limit printed.

In [ ]:
for key, value in an.dynamics.items():
    print(f"{key:32s} {value}")
fa.plot_acceleration(an); plt.show()
fa.plot_orientation(an); plt.show()

## 6 · Sound, GPS and correlations

The acoustic level is relative millivolts, not decibels. The GPS drift compares fixes before
release with fixes after landing. The correlation panels check the barometer against the
standard atmosphere, and show why a temperature lapse rate over 30 m is not measurable with this
sensor.

In [ ]:
for section in ("sound", "gps", "environment"):
    print(f"--- {section}")
    for key, value in getattr(an, section).items():
        print(f"{key:36s} {value}")
fa.plot_sound(an); plt.show()
fa.plot_gps(an); plt.show()
fa.plot_correlations(an); plt.show()

## 7 · The onboard log against what the ground heard

With the SD log as the primary input, every packet number missing from the ground log is a packet
the radio lost. Losses during the descent matter most: the descent is only about twenty packets.

In [ ]:
if an.comparison:
    for key, value in an.comparison.items():
        print(f"{key:48s} {value}")
else:
    print("no COMPARE_LOG given")

## 8 · Write everything out

Every figure as a PNG, `summary.md` with every number above in tables ready for the report, and
`analysis.json` with the raw values.

In [ ]:
out = repo_path(OUTPUT_DIR)
figures = fa.save_figures(an, out)
(out / "summary.md").write_text(fa.summary_markdown(an, figures), encoding="utf-8")
import json
(out / "analysis.json").write_text(json.dumps(an.as_dict(), indent=2, default=str), encoding="utf-8")
plt.close("all")
print(f"wrote {len(figures)} figures, summary.md and analysis.json to {out}")

## Before this goes in the report

- [ ] The paths in **Configuration** point at the real logs, not `test-data/synthetic-flight/`.
- [ ] The data-quality notes were read, and any multi-session warning is understood.
- [ ] The release and landing times look right on the descent graph.
- [ ] Acceleration peaks are described as lower bounds; yaw as relative.
- [ ] If `MASS_KG` was a guess, the drag coefficient is labelled as depending on it.